In [ ]:
import pandas as pd
data=pd.read_csv('/Users/deepandee/Desktop/Download the modified dataset with class 2 samples')
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

data['AI_model_similarity'] = data['AI_model_similarity'].map({'same': 1, 'different': 0})
data['proximity_behavior'] = data['proximity_behavior'].map({'close': 2, 'medium': 1, 'far': 0})
data['reaction_to_praise'] = data['reaction_to_praise'].astype('category').cat.codes

data=data.sample(frac=1, random_state=42)
train_len=int(len(data)*0.8)
x_train = data.drop(columns=['label', 'final_label'])
y_train = data['final_label']

x_test = data.drop(columns=['label', 'final_label'])
y_test = data['final_label']
scale=StandardScaler()
x_train_scal=scale.fit_transform(x_train)
x_test_scal=scale.transform(x_test)
from sklearn.model_selection import cross_val_score
import xgboost as xgb
from sklearn.metrics import classification_report

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

xgb_model.fit(x_train_scal, y_train)

y_pred_xgb = xgb_model.predict(x_test_scal)

print("XGBoost Classification Report:\n")
print(classification_report(y_test, y_pred_xgb))

scores = cross_val_score(xgb_model, x_train_scal, y_train, cv=25)
print("Cross-Validation Scores:", scores)
print("Mean CV Score:", scores.mean())